# Learning Module: Fuzzy Logic RKA 25

## 1. Introduction

Classical (*crisp*) logic only knows two values: **true (1)** or **false (0)**. Yet many everyday concepts are **vague**: "hot", "tall", "good service".

Example: if "hot" is defined as temperature ≥ 30°C, then 29.9°C is **not hot** at all. That makes little sense.

**Fuzzy logic** (Lotfi Zadeh, 1965) allows a **degree of membership** μ(x) ∈ [0, 1]. A temperature of 29.9°C can be "hot" to a degree of 0.95.

In this module we implement everything from scratch (only `numpy` and `matplotlib`):

| Part | Content |
|---|---|
| Basics | Membership functions, linguistic variables, fuzzy set operations |
| Inference 1 | **Mamdani** (fuzzy set output, centroid defuzzification) |
| Inference 2 | **Sugeno** (constant output, weighted average) |
| Inference 3 | **Tsukamoto** (monotonic output sets, weighted average) |
| Comparison | Output table & control surfaces of all three methods |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
temp = np.linspace(15, 45, 301)

crisp_hot = (temp >= 30).astype(float)
fuzzy_hot = np.clip((temp - 25) / 10, 0, 1)   # rises linearly from 25°C to 35°C

plt.figure(figsize=(7, 3))
plt.plot(temp, crisp_hot, label='Crisp: hot if temperature ≥ 30°C', lw=2)
plt.plot(temp, fuzzy_hot, label='Fuzzy: degree of hotness', lw=2)
plt.xlabel('Temperature (°C)'); plt.ylabel('μ hot'); plt.legend(); plt.grid(alpha=.3)
plt.title('Crisp set vs fuzzy set')
plt.show()

for t in [29.9, 30, 32]:
    print(f"Temperature {t}°C -> crisp = {float(t >= 30):.0f}, fuzzy = {np.clip((t - 25) / 10, 0, 1):.2f}")

## 2. Membership Functions

A membership function μ(x) maps an input value to a degree of membership in [0, 1]. The three most common shapes:

| Function | Parameters | Shape |
|---|---|---|
| **Triangular** (`trimf`) | a, b, c | rises from a to b, falls from b to c |
| **Trapezoidal** (`trapmf`) | a, b, c, d | rises a→b, flat (μ=1) b→c, falls c→d |
| **Gaussian** (`gaussmf`) | mean, sigma | bell curve |

Trapezoid formula:

$$
\mu(x) = \begin{cases} 0 & x \le a \text{ or } x \ge d \\ \frac{x-a}{b-a} & a < x < b \\ 1 & b \le x \le c \\ \frac{d-x}{d-c} & c < x < d \end{cases}
$$

A triangle is a special case of a trapezoid with b = c. If a = b (or c = d), that side is vertical and forms a **shoulder**, which suits sets at the edges of the universe.

In [ ]:
def trapmf(x, a, b, c, d):
    x = np.asarray(x, dtype=float)
    y = np.zeros_like(x)
    if b > a:
        y = np.where((x > a) & (x < b), (x - a) / (b - a), y)   # rising edge
    y = np.where((x >= b) & (x <= c), 1.0, y)                  # flat top
    if d > c:
        y = np.where((x > c) & (x < d), (d - x) / (d - c), y)   # falling edge
    return y if y.ndim else float(y)


def trimf(x, a, b, c):
    return trapmf(x, a, b, b, c)


def gaussmf(x, mean, sigma):
    y = np.exp(-((np.asarray(x, dtype=float) - mean) ** 2) / (2 * sigma ** 2))
    return y if y.ndim else float(y)

In [ ]:
x = np.linspace(0, 10, 1001)

plt.figure(figsize=(7, 3))
plt.plot(x, trimf(x, 1, 3, 5), label='trimf(1, 3, 5)')
plt.plot(x, trapmf(x, 4, 5, 7, 8), label='trapmf(4, 5, 7, 8)')
plt.plot(x, gaussmf(x, 8, 0.8), label='gaussmf(8, 0.8)')
plt.plot(x, trapmf(x, 0, 0, 0.5, 2), '--', label='left shoulder trapmf(0, 0, 0.5, 2)')
plt.ylim(-0.05, 1.05); plt.grid(alpha=.3); plt.legend(fontsize=8)
plt.title('Membership function shapes')
plt.show()

print("trimf(2, 1, 3, 5)      =", trimf(2, 1, 3, 5))
print("trapmf(6, 4, 5, 7, 8)  =", trapmf(6, 4, 5, 7, 8))

## 3. Linguistic Variables and Fuzzification

### 3.1 Case Study: Restaurant Tipping

We design a fuzzy system that determines the **tip (%)** based on the quality of **service** and **food**. This is the classic example also used in the `scikit-fuzzy` documentation, so the results can be cross-checked.

| Variable | Universe | Fuzzy sets (trimf) |
|---|---|---|
| Service (input) | 0 – 10 | poor (0,0,5), average (0,5,10), good (5,10,10) |
| Food (input) | 0 – 10 | poor (0,0,5), average (0,5,10), delicious (5,10,10) |
| Tip (output) | 0 – 25 % | low (0,0,13), medium (0,13,25), high (13,25,25) |

A **linguistic variable** is a variable whose values are words (poor, average, good), and each word is defined by a fuzzy set.

In [ ]:
service_sets = {
    'poor':    lambda v: trimf(v, 0, 0, 5),
    'average': lambda v: trimf(v, 0, 5, 10),
    'good':    lambda v: trimf(v, 5, 10, 10),
}
food_sets = {
    'poor':      lambda v: trimf(v, 0, 0, 5),
    'average':   lambda v: trimf(v, 0, 5, 10),
    'delicious': lambda v: trimf(v, 5, 10, 10),
}
tip_sets = {
    'low':    lambda v: trimf(v, 0, 0, 13),
    'medium': lambda v: trimf(v, 0, 13, 25),
    'high':   lambda v: trimf(v, 13, 25, 25),
}

x_in = np.linspace(0, 10, 1001)
z_tip = np.linspace(0, 25, 2501)    # output universe (resolution 0.01)


def plot_sets(universe, sets, title, ax, marker=None):
    for name, mf in sets.items():
        ax.plot(universe, mf(universe), lw=2, label=name)
    if marker is not None:
        ax.axvline(marker, color='k', ls='--', lw=1)
    ax.set_title(title); ax.set_ylim(-0.05, 1.05); ax.grid(alpha=.3); ax.legend(fontsize=8)


fig, axes = plt.subplots(1, 3, figsize=(14, 3))
plot_sets(x_in, service_sets, 'Service', axes[0])
plot_sets(x_in, food_sets, 'Food', axes[1])
plot_sets(z_tip, tip_sets, 'Tip (%)', axes[2])
plt.tight_layout(); plt.show()

### 3.2 Fuzzification

**Fuzzification** turns a crisp input value into degrees of membership in every fuzzy set.

In [ ]:
def fuzzify(value, sets):
    return {name: float(mf(value)) for name, mf in sets.items()}


print("Service = 9.8 ->", fuzzify(9.8, service_sets))
print("Food    = 6.5 ->", fuzzify(6.5, food_sets))

Food = 6.5 is a member of **average** (0.7) and **delicious** (0.3) at the same time. This is the essence of fuzzy logic: one value can belong to several sets with different degrees.

## 4. Fuzzy Set Operations

Logical operators are extended to degrees of membership (standard Zadeh operators):

| Operation | Logic | Formula |
|---|---|---|
| Intersection (AND) | A ∧ B | μ = min(μA, μB) |
| Union (OR) | A ∨ B | μ = max(μA, μB) |
| Complement (NOT) | ¬A | μ = 1 − μA |

In [ ]:
mu_a = trimf(x_in, 1, 4, 7)
mu_b = trimf(x_in, 3, 6, 9)

fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, (title, result) in zip(axes, [
    ('AND: min(A, B)', np.minimum(mu_a, mu_b)),
    ('OR: max(A, B)', np.maximum(mu_a, mu_b)),
    ('NOT A: 1 - A', 1 - mu_a),
]):
    ax.plot(x_in, mu_a, '--', label='A'); ax.plot(x_in, mu_b, '--', label='B')
    ax.fill_between(x_in, result, alpha=.4, label='result')
    ax.set_title(title); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 5. Rule Base

A fuzzy inference system uses **IF–THEN** rules. For the tipping case:

| Rule | IF | THEN |
|---|---|---|
| R1 | service is **poor** OR food is **poor** | tip is **low** |
| R2 | service is **average** | tip is **medium** |
| R3 | service is **good** OR food is **delicious** | tip is **high** |

The degree of truth of the IF part is called the **α-predicate** (*firing strength*). OR is computed with `max`, AND with `min`.

We write each rule as a pair `(α function, output label)`, where the α function receives the fuzzified values of all inputs.

In [ ]:
tip_rules = [
    (lambda s, f: max(s['poor'], f['poor']),      'low'),     # R1
    (lambda s, f: s['average'],                   'medium'),  # R2
    (lambda s, f: max(s['good'], f['delicious']), 'high'),    # R3
]


def fire_rules(rules, *degrees):
    # Return a list of (alpha, label) for every rule
    return [(rule(*degrees), label) for rule, label in rules]


s_deg = fuzzify(9.8, service_sets)
f_deg = fuzzify(6.5, food_sets)
alphas = fire_rules(tip_rules, s_deg, f_deg)
for i, (a, label) in enumerate(alphas, 1):
    print(f"R{i}: α = {a:.3f}  -> tip {label}")

## 6. Mamdani Inference

### 6.1 Steps

The Mamdani (*max-min*) method produces a **fuzzy set** as output and then converts it into a crisp number:

1. **Fuzzify** the inputs.
2. **Evaluate the rules**: compute α for every rule.
3. **Implication (min)**: clip each rule's output set at height α → `min(α, μ_output(z))`.
4. **Aggregation (max)**: combine all clipped sets → point-wise `max`.
5. **Defuzzification**: turn the aggregated set into a single number.

### 6.2 Defuzzification

| Method | Formula | Notes |
|---|---|---|
| **Centroid** (Center of Gravity) | $z^* = \dfrac{\int z\,\mu(z)\,dz}{\int \mu(z)\,dz}$ | most common, smooth result |
| **Mean of Maximum (MOM)** | average of the z values with maximal μ | simpler, can "jump" |

The integrals are computed numerically on the discrete universe `z_tip` (resolution 0.01).

In [ ]:
def centroid(z, mu):
    total = np.sum(mu)
    return float(np.sum(z * mu) / total) if total > 0 else float('nan')


def mean_of_maximum(z, mu):
    if mu.max() == 0:
        return float('nan')
    return float(np.mean(z[np.isclose(mu, mu.max())]))


def mamdani(alphas, out_sets, z):
    aggregated = np.zeros_like(z)
    for alpha, label in alphas:
        clipped = np.minimum(alpha, out_sets[label](z))   # min implication
        aggregated = np.maximum(aggregated, clipped)      # max aggregation
    return centroid(z, aggregated), aggregated

In [ ]:
tip_mamdani, agg = mamdani(alphas, tip_sets, z_tip)
print(f"Mamdani tip (centroid) = {tip_mamdani:.2f} %")
print(f"Mamdani tip (MOM)      = {mean_of_maximum(z_tip, agg):.2f} %")

fig, ax = plt.subplots(figsize=(8, 3.5))
for name, mf in tip_sets.items():
    ax.plot(z_tip, mf(z_tip), '--', lw=1, label=name)
ax.fill_between(z_tip, agg, alpha=.5, color='orange', label='aggregated')
ax.axvline(tip_mamdani, color='k', lw=2, label=f'centroid = {tip_mamdani:.2f}')
ax.set_xlabel('Tip (%)'); ax.set_ylabel('μ'); ax.grid(alpha=.3); ax.legend(fontsize=8)
ax.set_title('Mamdani: service = 9.8, food = 6.5')
plt.show()

> With the same inputs, `scikit-fuzzy` (the *tipping problem* example in its documentation) gives ≈19.85%. Our implementation produces the same result.

## 7. Sugeno Inference (Takagi–Sugeno–Kang)

### 7.1 Concept

The main difference from Mamdani: **each rule's output is a crisp function**, not a fuzzy set.

- **Zero-order Sugeno**: constant output → `IF ... THEN z = k`
- **First-order Sugeno**: linear output → `IF ... THEN z = p·x + q·y + r`

The final output is a **weighted average**, so **no defuzzification is needed**:

$$
z^* = \frac{\sum_i \alpha_i \, z_i}{\sum_i \alpha_i}
$$

### 7.2 Zero-Order Sugeno Implementation

Each label's constant is set to the **centroid** of the corresponding Mamdani output set so the two can be compared:
- low = (0 + 0 + 13) / 3 ≈ 4.33
- medium = (0 + 13 + 25) / 3 ≈ 12.67
- high = (13 + 25 + 25) / 3 = 21

In [ ]:
tip_constants = {'low': 13 / 3, 'medium': 38 / 3, 'high': 21.0}


def sugeno(alphas, constants):
    num = sum(alpha * constants[label] for alpha, label in alphas)
    den = sum(alpha for alpha, _ in alphas)
    return num / den if den > 0 else float('nan')


tip_sugeno = sugeno(alphas, tip_constants)
for a, label in alphas:
    print(f"α = {a:.3f}, z = {tip_constants[label]:.2f}  ({label})")
print(f"\nSugeno tip = {tip_sugeno:.2f} %")

## 8. Tsukamoto Inference

### 8.1 Concept

In Tsukamoto, every output set must be **monotonic** (always increasing or always decreasing). For each rule:

1. Compute α.
2. Find the z that gives μ_output(z) = α, i.e. the **inverse** of the membership function: z = μ⁻¹(α).
3. The final output = weighted average of all z.

Because the triangular **medium** set is not monotonic (it rises then falls), Tsukamoto only uses two output sets:

| Set | Function | Inverse |
|---|---|---|
| low (decreasing) | μ(z) = (25 − z) / 25 | z = 25 − 25α |
| high (increasing) | μ(z) = z / 25 | z = 25α |

The rule base is adjusted to:

| Rule | IF | THEN |
|---|---|---|
| R1 | service is **poor** OR food is **poor** | tip is **low** |
| R2 | service is **good** OR food is **delicious** | tip is **high** |

In [ ]:
tsk_sets = {
    'low':  lambda z: np.clip((25 - np.asarray(z)) / 25, 0, 1),
    'high': lambda z: np.clip(np.asarray(z) / 25, 0, 1),
}
tsk_inverse = {
    'low':  lambda a: 25 - 25 * a,
    'high': lambda a: 25 * a,
}
tsk_rules = [
    (lambda s, f: max(s['poor'], f['poor']),      'low'),
    (lambda s, f: max(s['good'], f['delicious']), 'high'),
]


def tsukamoto(alphas, inverse):
    zs = [(alpha, inverse[label](alpha)) for alpha, label in alphas]
    den = sum(alpha for alpha, _ in zs)
    return sum(alpha * z for alpha, z in zs) / den if den > 0 else float('nan')


tsk_alphas = fire_rules(tsk_rules, s_deg, f_deg)
for a, label in tsk_alphas:
    print(f"α = {a:.3f} -> z = μ⁻¹(α) = {tsk_inverse[label](a):.2f}  ({label})")
tip_tsukamoto = tsukamoto(tsk_alphas, tsk_inverse)
print(f"\nTsukamoto tip = {tip_tsukamoto:.2f} %")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for (name, mf), color in zip(tsk_sets.items(), ['tab:blue', 'tab:red']):
    ax.plot(z_tip, mf(z_tip), color=color, lw=2, label=name)
for a, label in tsk_alphas:
    if a > 0:
        z_i = tsk_inverse[label](a)
        ax.plot([z_i, z_i], [0, a], 'k:'); ax.plot(z_i, a, 'ko')
        ax.annotate(f'α={a:.2f}, z={z_i:.1f}', (z_i, a), textcoords='offset points', xytext=(-60, 8))
ax.set_xlabel('Tip (%)'); ax.set_ylabel('μ'); ax.grid(alpha=.3); ax.legend()
ax.set_title('Tsukamoto: z is obtained from the inverse of a monotonic membership function')
plt.show()

## 9. Comparing Mamdani, Sugeno, and Tsukamoto

### 9.1 Output Table

In [ ]:
def tip_all_methods(service, food):
    s, f = fuzzify(service, service_sets), fuzzify(food, food_sets)
    a = fire_rules(tip_rules, s, f)
    return (mamdani(a, tip_sets, z_tip)[0],
            sugeno(a, tip_constants),
            tsukamoto(fire_rules(tsk_rules, s, f), tsk_inverse))


samples = [(0, 0), (2, 8), (5, 5), (6.5, 6.5), (9.8, 6.5), (10, 10)]
print(f"{'Service':>8} {'Food':>6} | {'Mamdani':>8} {'Sugeno':>8} {'Tsukamoto':>10}")
print("-" * 49)
for sv, fv in samples:
    mam, sug, tsk = tip_all_methods(sv, fv)
    print(f"{sv:>8} {fv:>6} | {mam:>8.2f} {sug:>8.2f} {tsk:>10.2f}")

Things to notice:
- **(5, 5)**: Tsukamoto returns `nan` because **no rule fires** (every α = 0), since there is no rule for the "average" condition. This is an **edge case**: the rule base must cover the whole input space.
- **(6.5, 6.5)**: Tsukamoto gives only 7.5%, far below Mamdani and Sugeno. Only the "high" rule fires, with α = 0.3, so z = 25 × 0.3 = 7.5. In Tsukamoto, α **shifts the position** of z, while in Mamdani/Sugeno α only acts as a **weight**. As a result, weakly firing rules can produce counter-intuitive outputs.
- **(0, 0)** and **(10, 10)**: Mamdani and Sugeno are stuck at 4.33 and 21, while Tsukamoto reaches the edges of the universe (0 and 25).

### 9.2 Control Surface

The control surface shows the output for **every** input combination. It reveals how **smoothly** the output changes.

In [ ]:
grid = np.linspace(0, 10, 41)
surfaces = np.zeros((3, len(grid), len(grid)))
for i, fv in enumerate(grid):
    for j, sv in enumerate(grid):
        surfaces[:, i, j] = tip_all_methods(sv, fv)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, surf, title in zip(axes, surfaces, ['Mamdani', 'Zero-order Sugeno', 'Tsukamoto']):
    im = ax.contourf(grid, grid, surf, levels=20, cmap='viridis', vmin=0, vmax=25)
    ax.set_title(title); ax.set_xlabel('Service'); ax.set_ylabel('Food')
fig.colorbar(im, ax=axes, label='Tip (%)')
plt.show()

Mamdani and Sugeno change smoothly. The Tsukamoto surface **jumps sharply** around (5, 5): there both rules barely fire, so a small change in the input changes which rule dominates the weighted average.

### 9.3 Summary

| Aspect | Mamdani | Sugeno | Tsukamoto |
|---|---|---|---|
| Rule output | Fuzzy set | Constant / linear function | **Monotonic** fuzzy set |
| Defuzzification | Required (centroid, MOM, ...) | Not needed (weighted average) | Not needed (weighted average) |
| Computation | Heaviest (integral over the output universe) | Lightest | Light |
| Interpretability | High, rules read like human language | Medium, constants/coefficients are less intuitive | Medium, limited by the monotonic requirement |
| Output smoothness | Smooth | Smooth | Can jump when rule coverage is incomplete (see around (5, 5)) |
| Output range | Never reaches the edges of the universe (centroid effect) | Between the smallest and largest constant | Can reach the edges of the universe |
| Typical use | Expert systems, decisions based on human knowledge | Control, optimization, ANFIS | Systems with a monotonic input–output relationship |

> **Note:** with Mamdani, the tip never reaches 0% or 25% even for extreme inputs (0,0) or (10,10). This is because the centroid of a triangle always lies inside the triangle, never at its tip.

## 10. Exercises (Homework)
<font color="blue">Exercise 8</font>

### Homework — Fuzzy Washing Machine Duration

Design a fuzzy system that determines the **washing duration (minutes)** based on **dirt level** and **load weight**.

| Variable | Universe | Fuzzy sets |
|---|---|---|
| Dirt (input) | 0 – 100 % | low (0,0,50), medium (0,50,100), high (50,100,100) |
| Load (input) | 0 – 10 kg | light trapmf(0,0,2,5), medium trimf(2,5,8), heavy trapmf(5,8,10,10) |
| Duration (output) | 0 – 60 minutes | short (0,0,30), normal (10,30,50), long (30,60,60) |

**Rule base:**

| Rule | IF | THEN |
|---|---|---|
| R1 | dirt is **low** AND load is **light** | duration is **short** |
| R2 | dirt is **medium** OR load is **medium** | duration is **normal** |
| R3 | dirt is **high** AND load is **heavy** | duration is **long** |
| R4 | dirt is **high** AND load is **light** | duration is **normal** |
| R5 | dirt is **low** AND load is **heavy** | duration is **normal** |

**Tasks:**
1. Complete the fuzzy set definitions and the rule base (the `_____` parts).
2. Plot the membership functions of all variables.
3. Compute the duration with **Mamdani** and **zero-order Sugeno** for inputs `(20, 3)`, `(60, 6)`, `(90, 9)`.
4. Compare the results of both methods and explain the differences.

### 1. Fuzzy Sets

In [ ]:
dirt_sets = {
    'low':    lambda v: trimf(v, 0, 0, 50),
    'medium': lambda v: _____,
    'high':   lambda v: _____,
}
load_sets = {
    'light':  lambda v: trapmf(v, 0, 0, 2, 5),
    'medium': lambda v: _____,
    'heavy':  lambda v: _____,
}
duration_sets = {
    'short':  lambda v: _____,
    'normal': lambda v: _____,
    'long':   lambda v: _____,
}

z_duration = np.linspace(0, 60, 6001)

### 2. Membership Function Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
plot_sets(np.linspace(0, 100, 1001), dirt_sets, 'Dirt (%)', axes[0])
plot_sets(np.linspace(0, 10, 1001), load_sets, 'Load (kg)', axes[1])
plot_sets(z_duration, duration_sets, 'Duration (minutes)', axes[2])
plt.tight_layout(); plt.show()

### 3. Rule Base

In [ ]:
wash_rules = [
    (lambda d, l: min(d['low'], l['light']), 'short'),    # R1
    (lambda d, l: _____,                     'normal'),   # R2
    (lambda d, l: _____,                     'long'),     # R3
    (lambda d, l: _____,                     'normal'),   # R4
    (lambda d, l: _____,                     'normal'),   # R5
]

# Zero-order Sugeno constants (centroid of each output set)
wash_constants = {'short': 10.0, 'normal': 30.0, 'long': 50.0}

### 4. Testing

In [ ]:
print(f"{'Dirt':>5} {'Load':>5} | {'Mamdani':>8} {'Sugeno':>8}")
print("-" * 33)
for dv, lv in [(20, 3), (60, 6), (90, 9)]:
    d, l = fuzzify(dv, dirt_sets), fuzzify(lv, load_sets)
    a = fire_rules(wash_rules, d, l)
    dur_mamdani = mamdani(a, duration_sets, z_duration)[0]
    dur_sugeno = _____
    print(f"{dv:>5} {lv:>5} | {dur_mamdani:>8.2f} {dur_sugeno:>8.2f}")

**Analysis (write your answer here):**
1. Compare the Mamdani and Sugeno results for the three inputs. Which one is larger/smaller, and why?
2. Why are the Sugeno constants `short = 10` and `long = 50`, rather than 0 and 60?
3. If you wanted to use Tsukamoto, which output sets would need to change? Why?